# CARLA Python API - Project 5: Cooperative Roadside Assistant (V2X / I2V)

This notebook contains exactly 3 sections:
1. **Infrastructure Sensing & World Transformation** (Theory, camera setups, and 2D-to-3D projection layout)
2. **V2X MQTT Server Architecture & Message Serialization** (Live MQTT client setup, network delay emulation, and protocol payloads)
3. **End-to-End System Integration & Occlusion Scenarios** (The runnable project execution with full evaluation metrics)

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/

In [1]:
import carla
import time
import random
import cv2
import queue
import threading
import math
import numpy as np

# ==============================================================================
# 1. CORE UTILITIES AND RSU TRANSFORMS
# ==============================================================================

def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

# Connect to CARLA Simulator
client = carla.Client("localhost", 2000)
client.set_timeout(20.0)

In [2]:
# Global simulated V2X message broker queue for synchronization
v2x_broker_channel = queue.Queue(maxsize=20)

def roadside_unit_camera_loop(rsu_sensor, target_actors, stop_event):
    """
    Simulates a smart infrastructure RSU camera tracking hidden VRUs.
    Handles both single actor instances and grouped arrays seamlessly.
    """
    frame_queue = queue.Queue(maxsize=1)
    rsu_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)
    
    # --- FIX: Dynamically handle single actor types passed by legacy base functions ---
    if not isinstance(target_actors, (list, tuple, set)):
        target_actors = [target_actors]
    
    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            v2x_payload = []
            
            for actor in target_actors:
                if actor is not None and actor.is_alive:
                    tf = actor.get_transform()
                    vel = actor.get_velocity()
                    
                    v2x_payload.append({
                        "id": actor.id,
                        "type": "Cyclist" if "bicycle" in actor.type_id else "Pedestrian",
                        "pos_x": tf.location.x,
                        "pos_y": tf.location.y,
                        "pos_z": tf.location.z,
                        "speed": 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
                    })
            
            if not v2x_broker_channel.full():
                v2x_broker_channel.put({
                    "timestamp": image.timestamp,
                    "objects": v2x_payload
                })
                
            # Render infrastructure visualization feed
            arr = np.frombuffer(image.raw_data, dtype=np.uint8)
            arr = np.reshape(arr, (image.height, image.width, 4))
            bgr_frame = arr[:, :, :3].copy()
            cv2.putText(bgr_frame, f"RSU INFRASTRUCTURE MESH: {len(v2x_payload)} ACTIVE TARGETS", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
            cv2.imshow("RSU Infrastructure Perception Feed", bgr_frame)
            cv2.waitKey(1)
            
        except queue.Empty:
            continue
            
    rsu_sensor.stop()
    cv2.destroyWindow("RSU Infrastructure Perception Feed")

In [ ]:
def run_assisted_scenario_v2(map_name, weather_preset, scenario_id):
    """
    Executes an enhanced V2X cooperative perception trial for exactly 30 seconds.
    Spawns a dense crowd of 6 fast-moving targets to test early wireless intervention metrics.
    """
    global v2x_hud_alert_triggered
    v2x_hud_alert_triggered = False
    
    print(f"\n" + "="*80)
    print(f"STARTING ENHANCED V2X ASSISTED TRIAL: Map={map_name} | Scenario={scenario_id}")
    print("="*80)
    
    while not v2x_broker_channel.empty():
        v2x_broker_channel.get()
        
    world = client.load_world(map_name)
    world.set_weather(weather_preset)
    blueprint_library = world.get_blueprint_library()
    spectator = world.get_spectator()
    
    settings = world.get_settings()
    original_settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.04
    world.apply_settings(settings)
    
    trial_actors = []
    vru_group = []
    cam_thread = None
    rsu_thread = None
    stop_signal = threading.Event()
    
    min_distance_to_target = float("inf")
    collision_detected = False
    speed_at_brake_trigger = 0.0
    brake_timestamp = None
    
    try:
        carla_map = world.get_map()
        spawn_points = carla_map.get_spawn_points()
        
        if "Town03" in map_name:
            ego_spawn_tf = carla.Transform(carla.Location(x=120.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
            rsu_spawn_tf = carla.Transform(carla.Location(x=145.0, y=145.0, z=8.0), carla.Rotation(pitch=-35.0, yaw=-135.0, roll=0.0))
            vru_direction = carla.Vector3D(0, -1, 0)
            
            # Synchronized base coordinate set for the 6-target crosswalk crowd
            base_x, base_y, base_z = 144.0, 141.0, 2.5
            vru_count = 6
        else:
            ego_spawn_tf = spawn_points[0]
            rsu_spawn_tf = carla.Transform(ego_spawn_tf.location + carla.Location(x=20, y=10, z=8.0), carla.Rotation(pitch=-35.0, yaw=-90.0))
            vru_direction = carla.Vector3D(-1, 0, 0)
            base_x, base_y, base_z = spawn_points[5].location.x, spawn_points[5].location.y, spawn_points[5].location.z + 1.5
            vru_count = 2

        # Spawn Ego vehicle safely
        ego_vehicle = world.try_spawn_actor(blueprint_library.filter("vehicle.tesla.model3")[0], ego_spawn_tf)
        if ego_vehicle is None:
            ego_spawn_tf.location.x += 2.0
            ego_vehicle = world.spawn_actor(blueprint_library.filter("vehicle.tesla.model3")[0], ego_spawn_tf)
        trial_actors.append(ego_vehicle)
        
        # Filter pedestrian and vehicle assets
        ped_blueprints = blueprint_library.filter("walker.pedestrian.*")
        bike_blueprints = blueprint_library.filter("vehicle.bh.bicycle") or blueprint_library.filter("vehicle.diamondback.century") or blueprint_library.filter("vehicle.*")
        
        ped_bp = ped_blueprints[0]
        bike_bp = bike_blueprints[0] if bike_blueprints else ped_bp
        
        # --- ADAPTIVE MULTI-PASSENGER SPAWNING BLOCK (6 SCALED TARGETS) ---
        for i in range(vru_count):
            # Mix in a cyclist asset at index 2 to create cross-traffic diversity
            bp = bike_bp if (i == 2 and "Town03" in map_name) else ped_blueprints[i % len(ped_blueprints)]
            vru = None
            
            for attempt in range(10):
                row = i // 3
                col = i % 3
                shift_x = base_x + (col * 2.0) + (attempt * 0.3)
                shift_y = base_y + (row * 1.5) + (attempt * 0.1)
                spawn_tf = carla.Transform(carla.Location(x=shift_x, y=shift_y, z=base_z), carla.Rotation(yaw=-90.0))
                
                vru = world.try_spawn_actor(bp, spawn_tf)
                if vru is not None:
                    trial_actors.append(vru)
                    vru_group.append(vru)
                    break
            
            if vru is None:
                nav_loc = world.get_random_location_from_navigation()
                if nav_loc is not None:
                    vru = world.try_spawn_actor(bp, carla.Transform(nav_loc + carla.Location(z=1.0), carla.Rotation(yaw=0.0)))
                    if vru is not None:
                        trial_actors.append(vru)
                        vru_group.append(vru)

        # Setup Sensors
        ego_camera = world.spawn_actor(blueprint_library.find("sensor.camera.rgb"), carla.Transform(carla.Location(x=2.0, z=1.3)), attach_to=ego_vehicle)
        trial_actors.append(ego_camera)
        
        rsu_sensor = world.spawn_actor(blueprint_library.find("sensor.camera.rgb"), rsu_spawn_tf)
        trial_actors.append(rsu_sensor)
        
        # Launch Monitoring Threads
        cam_thread = threading.Thread(target=local_onboard_camera_loop, args=(ego_camera, stop_signal, "V2X-COOPERATIVE"), daemon=True)
        rsu_thread = threading.Thread(target=roadside_unit_camera_loop, args=(rsu_sensor, vru_group, stop_signal), daemon=True)
        cam_thread.start()
        rsu_thread.start()
        
        world.tick()
        time.sleep(0.6)
        
        # --- SPRINTING VELOCITY ACTIVATION ---
        for vru in vru_group:
            if "walker" not in vru.type_id:
                vru.apply_control(carla.VehicleControl(throttle=0.45)) 
            else:
                vru.apply_control(carla.WalkerControl(direction=vru_direction, speed=4.2 + random.uniform(-0.4, 0.4)))
        
        target_throttle, target_brake = 0.50, 0.0
        
        # --- FIXED 30-SECOND SIMULATION TICK MATRIX ---
        for frame in range(750):
            world.tick()
            current_sim_time = frame * 0.04
            
            ego_tf = ego_vehicle.get_transform()
            move_spectator_to(ego_tf, spectator, distance=15.0, z=6.0, pitch=-22.0)
            
            # Monitor closest distance across the crowd cluster
            current_closest = float("inf")
            for vru in vru_group:
                if vru.is_alive:
                    dist = ego_tf.location.distance(vru.get_transform().location)
                    if dist < current_closest: current_closest = dist
            
            if current_closest < min_distance_to_target: min_distance_to_target = current_closest
            if current_closest < 2.10: collision_detected = True
            
            vel = ego_vehicle.get_velocity()
            speed_kmh = 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
            
            # --- COMPREHENSIVE V2X FUSION & CONSOLE LOGGING ---
            v2x_alert_active = False
            if not v2x_broker_channel.empty():
                latest_packet = v2x_broker_channel.get()
                
                for obj in latest_packet["objects"]:
                    obj_loc = carla.Location(obj["pos_x"], obj["pos_y"], obj["pos_z"])
                    dist_to_ego = ego_tf.location.distance(obj_loc)
                    
                    # Intercept trajectories inside a 32m warning radius
                    if dist_to_ego < 32.0:
                        v2x_alert_active = True
                        print(f"[📡 V2X LOG] Ingress Threat ID {obj['id']} ({obj['type']}) | Range: {dist_to_ego:.1f}m | Target Speed: {obj['speed']:.1f} km/h")
            
            v2x_hud_alert_triggered = v2x_alert_active
            
            if v2x_alert_active:
                if target_brake == 0.0:
                    brake_timestamp = current_sim_time
                    speed_at_brake_trigger = speed_kmh
                    print(f"[🚨 V2X POLICY] Cooperative early deceleration command initialized at t = {brake_timestamp:.2f}s")
                target_throttle, target_brake = 0.0, 0.70
                world.debug.draw_string(ego_tf.location + carla.Location(z=2.8), "📡 V2X NETWORK OVERLAY BRAKE", life_time=0.04, color=carla.Color(0,255,0))
            else:
                world.debug.draw_string(ego_tf.location + carla.Location(z=2.5), "CRUISE CONTROL ACTIVE", life_time=0.04, color=carla.Color(0,255,0))
                
            ego_vehicle.apply_control(carla.VehicleControl(throttle=float(target_throttle), brake=float(target_brake)))
            time.sleep(0.02)
            
    finally:
        stop_signal.set()
        if cam_thread is not None:
            cam_thread.join(timeout=1.5)
        if rsu_thread is not None:
            rsu_thread.join(timeout=1.5)
        world.apply_settings(original_settings)
        safe_destroy(trial_actors)
        
    return {"collision": collision_detected, "min_dist": min_distance_to_target, "brake_time": brake_timestamp, "trigger_speed": speed_at_brake_trigger}

In [ ]:
# ==============================================================================
# CELL 6: RUN BATCH EXECUTION PIPELINE WITH V2 ENHANCEMENTS
# ==============================================================================

# Execute V2X Enhanced Configurations (Assisted V2)
# --- FIX: Changed run_assisted_scenario to run_assisted_scenario_v2 ---
assisted_r1 = run_assisted_scenario_v2("Town03", carla.WeatherParameters.ClearNoon, 1)
assisted_r2 = run_assisted_scenario_v2("Town05", carla.WeatherParameters.HardRainSunset, 2)

# Generate Quantitative Benchmarking Table Output
print("\n" + "="*90)
print("                    PROJECT 5 DELIVERABLE: COOPERATIVE PERCEPTION REPORT                   ")
print("="*90)
print(f"{'Performance Metric':<32} | {'S1 Town03 (Base)':<16} | {'S1 Town03 (V2X)':<15} | {'S2 Town05 (Base)':<16} | {'S2 Town05 (V2X)':<15}")
print("-"*90)

o_a1 = "💥 COLLISION" if assisted_r1["collision"] else "✅ AVOIDED"
o_a2 = "💥 COLLISION" if assisted_r2["collision"] else "✅ AVOIDED"
print(f"{'Safety Validation Outcome':<32} | {o_a1:<15} | {o_a2:<15}")

print(f"{'Minimum Spatial Proximity (m)':<32} | {assisted_r1['min_dist']:<15.2f} | {assisted_r2['min_dist']:<15.2f}")

t_a1 = f"{assisted_r1['brake_time']:.2f}s" if assisted_r1['brake_time'] else "N/A"
t_a2 = f"{assisted_r2['brake_time']:.2f}s" if assisted_r2['brake_time'] else "N/A"
print(f"{'Perception Brake Trigger Time':<32} | {t_a1:<15} | {t_a2:<15}")

print("="*90)


LAUNCHING UNASSISTED MULTI-PASSENGER SCENARIO 1: Map=Town03
[📷 CAM LOCAL DETECT] Pedestrian emerged! Panic braking initiated at t = 2.80s


KeyboardInterrupt: 